In [1]:
import pandas as pd

In [ ]:
def run_etl_pipeline(account_profiles_df, fraud_patterns_df, network_edges_df, time_series_stats_df, transactions_df):

    # Make copies to avoid modifying original loaded dataframes if needed elsewhere
    account_profiles_df = account_profiles_df.copy()
    fraud_patterns_df = fraud_patterns_df.copy()
    network_edges_df = network_edges_df.copy()
    time_series_stats_df = time_series_stats_df.copy()
    transactions_df = transactions_df.copy()

    # --- 1. Datetime Conversion ---
    time_series_stats_df['hour'] = pd.to_datetime(time_series_stats_df['hour'])
    transactions_df['timestamp'] = pd.to_datetime(transactions_df['timestamp'])

    # --- 2. Missing Value Handling ---
    # Using .loc for explicit copy modification to avoid FutureWarning
    network_edges_df.loc[:, 'ring_id'] = network_edges_df['ring_id'].fillna('Unknown')
    transactions_df.loc[:, 'fraud_pattern'] = transactions_df['fraud_pattern'].fillna('No Fraud Pattern')

    # --- 3. Date Feature Extraction ---
    transactions_df.loc[:, 'transaction_year'] = transactions_df['timestamp'].dt.year
    transactions_df.loc[:, 'transaction_month'] = transactions_df['timestamp'].dt.month
    transactions_df.loc[:, 'transaction_month_name'] = transactions_df['timestamp'].dt.month_name()
    transactions_df.loc[:, 'transaction_day_name'] = transactions_df['timestamp'].dt.day_name()

    time_series_stats_df.loc[:, 'stats_year'] = time_series_stats_df['hour'].dt.year
    time_series_stats_df.loc[:, 'stats_month'] = time_series_stats_df['hour'].dt.month
    time_series_stats_df.loc[:, 'stats_month_name'] = time_series_stats_df['hour'].dt.month_name()
    time_series_stats_df.loc[:, 'stats_day_name'] = time_series_stats_df['hour'].dt.day_name()

    # --- 4. Boolean Conversion ---
    bool_cols_account_profiles = ['is_high_risk', 'has_2fa', 'is_fraudster']
    for col in bool_cols_account_profiles:
        account_profiles_df[col] = account_profiles_df[col].astype(bool)

    network_edges_df['both_fraud'] = (
        network_edges_df['both_fraud'].astype(bool)
    )

    bool_cols_transactions = ['card_present', 'device_known', 'is_foreign_txn', 'has_2fa', 'is_fraud']
    for col in bool_cols_transactions:
        transactions_df[col] = transactions_df[col].astype(bool)

    time_series_stats_df['is_weekend'] = (
        time_series_stats_df['is_weekend'].astype(bool)
    )

    return account_profiles_df, fraud_patterns_df, network_edges_df, time_series_stats_df, transactions_df

# Extracting Datasets

In [4]:
# Define paths to raw CSV files
raw_data_path = '../data/raw/'


def load_dataset(path):
    df = pd.read_csv(f'{raw_data_path}{path}')
    return df

# Load all raw CSV files
account_profiles_df = load_dataset("account_profiles.csv")
fraud_patterns_df = load_dataset("fraud_patterns.csv")
network_edges_df = load_dataset("network_edges.csv")
time_series_stats_df = load_dataset("time_series_stats.csv")
transactions_df = load_dataset("transactions.csv")

print("All raw CSV files loaded.")

All raw CSV files loaded.


# Execute ETL Pipeline

In [7]:
# Call the ETL pipeline function with the raw DataFrames
account_profiles_df_processed, fraud_patterns_df_processed, network_edges_df_processed, time_series_stats_df_processed, transactions_df_processed = run_etl_pipeline(
    account_profiles_df, fraud_patterns_df, network_edges_df, time_series_stats_df, transactions_df
)

print("ETL pipeline executed. Processed DataFrames are available.")

TypeError: Invalid value '[False False False ... False False False]' for dtype 'int64'

# Verify Transformed Outputs

In [ ]:
print('--- Account Profiles DataFrame (Processed) ---')
account_profiles_df_processed.info()
display(account_profiles_df_processed.head())

print('\n--- Network Edges DataFrame (Processed) ---')
network_edges_df_processed.info()
display(network_edges_df_processed.head())

print('\n--- Time Series Stats DataFrame (Processed) ---')
time_series_stats_df_processed.info()
display(time_series_stats_df_processed.head())

print('\n--- Transactions DataFrame (Processed) ---')
transactions_df_processed.info()
display(transactions_df_processed.head())